# ANP — Profiling do `ca-2016-01.csv`

Notebook exploratório do primeiro arquivo da série histórica de preços de combustíveis da ANP.

Objetivos:
- validar estrutura e qualidade do arquivo;
- documentar achados que influenciam a ingestão PostgreSQL;
- explorar período, produtos, cobertura geográfica e preços;
- testar a hipótese de chave natural `CNPJ + Produto + Data da Coleta`.

O notebook não faz parte do pipeline de produção. Ele é um artefato reproduzível de análise e Data Quality.


## Configuração

Defina a variável de ambiente `ANP_CSV_PATH` apontando para `ca-2016-01.csv`.

Exemplo no PowerShell:

```powershell
$env:ANP_CSV_PATH="F:\Estudos DataBricks\Projeto - Data Engineering Portfolio\fontes de dados\ANP - Serie Historica de combustiveis\ca-2016-01.csv"
```


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

source_path = Path(os.environ["ANP_CSV_PATH"])
assert source_path.is_file(), f"Arquivo não encontrado: {source_path}"

source_path


In [ ]:
df = pd.read_csv(
    source_path,
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
)

print(f"linhas={len(df):,}")
print(f"colunas={len(df.columns)}")
df.head()


## Estrutura e completude


In [ ]:
profile = pd.DataFrame(
    {
        "dtype_origem": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "distintos": df.nunique(dropna=True),
    }
).sort_values("pct_nulos", ascending=False)

profile


In [ ]:
print("Duplicatas de linha completa:", int(df.duplicated().sum()))


## Conversões técnicas usadas na ingestão PostgreSQL


In [ ]:
typed = df.copy()

typed["Data da Coleta"] = pd.to_datetime(
    typed["Data da Coleta"],
    format="%d/%m/%Y",
    errors="raise",
)

for column in ["Valor de Venda", "Valor de Compra"]:
    typed[column] = pd.to_numeric(
        typed[column].str.replace(",", ".", regex=False),
        errors="coerce",
    )

typed[["Data da Coleta", "Valor de Venda", "Valor de Compra"]].dtypes


## Período coberto


In [ ]:
periodo = pd.Series(
    {
        "data_min": typed["Data da Coleta"].min(),
        "data_max": typed["Data da Coleta"].max(),
        "dias_distintos": typed["Data da Coleta"].nunique(),
    }
)
periodo


## Produtos


In [ ]:
produtos = (
    typed.groupby("Produto", dropna=False)
    .size()
    .rename("registros")
    .sort_values(ascending=False)
)
produtos


In [ ]:
produtos.plot(kind="bar", title="Registros por produto")
plt.ylabel("Registros")
plt.xlabel("Produto")
plt.tight_layout()
plt.show()


## Cobertura geográfica


In [ ]:
por_uf = (
    typed.groupby("Estado - Sigla")
    .agg(
        registros=("Produto", "size"),
        municipios=("Municipio", "nunique"),
        revendas=("CNPJ da Revenda", "nunique"),
    )
    .sort_values("registros", ascending=False)
)

por_uf.head(27)


In [ ]:
por_uf["registros"].plot(kind="bar", title="Registros por UF")
plt.ylabel("Registros")
plt.xlabel("UF")
plt.tight_layout()
plt.show()


## Estatísticas de preço


In [ ]:
precos_por_produto = (
    typed.groupby("Produto")["Valor de Venda"]
    .agg(["count", "mean", "median", "min", "max", "std"])
    .round(3)
    .sort_values("mean")
)

precos_por_produto


In [ ]:
(
    typed.groupby(["Data da Coleta", "Produto"])["Valor de Venda"]
    .mean()
    .unstack()
    .plot(title="Preço médio de venda por produto ao longo do período")
)
plt.ylabel("Preço médio")
plt.xlabel("Data")
plt.tight_layout()
plt.show()


## Compra x venda


In [ ]:
margem = typed.dropna(subset=["Valor de Compra"]).copy()
margem["spread"] = margem["Valor de Venda"] - margem["Valor de Compra"]

margem.groupby("Produto")["spread"].agg(
    ["count", "mean", "median", "min", "max"]
).round(3)


## Bandeiras mais frequentes


In [ ]:
(
    typed["Bandeira"]
    .value_counts(dropna=False)
    .head(20)
    .rename("registros")
    .to_frame()
)


## Qualidade do CNPJ


In [ ]:
cnpj = typed["CNPJ da Revenda"]

pd.Series(
    {
        "registros": len(cnpj),
        "nulos": int(cnpj.isna().sum()),
        "com_espaco_esquerda": int(cnpj.fillna("").str.startswith(" ").sum()),
        "cnpjs_distintos": int(cnpj.nunique(dropna=True)),
    }
)


O espaço à esquerda deve ser tratado como característica do dado de origem.  
A ingestão PostgreSQL preserva o valor como `TEXT`; padronização de CNPJ fica para uma etapa posterior.


## Hipótese de chave natural


In [ ]:
candidate_key = ["CNPJ da Revenda", "Produto", "Data da Coleta"]

duplicated_candidate_key = typed.duplicated(
    subset=candidate_key,
    keep=False,
)

pd.Series(
    {
        "linhas": len(typed),
        "chaves_distintas": typed[candidate_key]
        .drop_duplicates()
        .shape[0],
        "linhas_em_chaves_duplicadas": int(duplicated_candidate_key.sum()),
    }
)


### Conclusão da hipótese

Se `linhas == chaves_distintas` e `linhas_em_chaves_duplicadas == 0`, a combinação
`CNPJ da Revenda + Produto + Data da Coleta` é única **neste arquivo**.

Isso não é suficiente para criar uma constraint permanente no banco: a hipótese deve ser validada nos demais períodos da série histórica.


## Resumo dos principais achados


In [ ]:
summary = {
    "rows": len(typed),
    "columns": len(typed.columns),
    "date_min": typed["Data da Coleta"].min().date().isoformat(),
    "date_max": typed["Data da Coleta"].max().date().isoformat(),
    "products": typed["Produto"].nunique(),
    "states": typed["Estado - Sigla"].nunique(),
    "municipalities": typed["Municipio"].nunique(),
    "full_row_duplicates": int(typed.duplicated().sum()),
    "candidate_key_duplicates": int(duplicated_candidate_key.sum()),
    "purchase_value_nulls": int(typed["Valor de Compra"].isna().sum()),
}

pd.Series(summary)
